# Day 10 — String & Date Functions

**What you will learn:**
- Essential string functions for text transformation and extraction
- Date/timestamp parsing, formatting, and arithmetic
- Pattern matching with `like` and `regexp_replace` / `regexp_extract`

**Exam relevance:** DataFrame API (30%) — string and date functions appear frequently, especially `to_date`, `date_format`, `datediff`, `split`, `regexp_replace`.

In [ ]:
from pyspark.sql.functions import *

df = spark.createDataFrame([
    (1, "  Alice Smith  ",  "Engineering",  "2022-03-15", "2024-06-01 08:30:00"),
    (2, "bob.jones",        "Marketing",    "2021-11-01", "2024-06-01 09:15:00"),
    (3, "CAROL O'BRIEN",    "Engineering",  "2020-07-22", "2024-06-01 07:45:00"),
    (4, "dave_123",         "Sales",        "2023-01-10", "2024-06-01 10:00:00"),
    (5, "Eve-Maria",        "Marketing",    "2021-05-30", "2024-06-01 08:00:00"),
], ["id", "raw_name", "dept", "hire_date_str", "login_ts_str"])
df.show(truncate=False)

## 1. String Cleaning

In [ ]:
df.select(
    col("raw_name"),
    trim(col("raw_name")).alias("trimmed"),
    upper(trim(col("raw_name"))).alias("upper"),
    lower(trim(col("raw_name"))).alias("lower"),
    length(trim(col("raw_name"))).alias("length"),
).show(truncate=False)

## 2. Substring, Split, Concat

In [ ]:
# substring(col, pos, len) — 1-indexed
df.select(
    col("raw_name"),
    substring(trim(col("raw_name")), 1, 3).alias("first_3"),
).show(truncate=False)

# split — returns ArrayType
df.select(
    trim(col("raw_name")).alias("name"),
    split(trim(col("raw_name")), " ").alias("parts"),
    split(trim(col("raw_name")), " ")[0].alias("first_name"),
).show(truncate=False)

# concat and concat_ws
df.select(
    concat(lit("EMP-"), col("id").cast("string")).alias("emp_code"),
    concat_ws(" | ", trim(col("raw_name")), col("dept")).alias("combined"),
).show(truncate=False)

## 3. Pattern Matching & Replacement

In [ ]:
# regexp_replace — replace all matches
df.select(
    col("raw_name"),
    regexp_replace(trim(col("raw_name")), r"[^a-zA-Z ]", "").alias("letters_only"),
).show(truncate=False)

# regexp_extract — capture a group
df.select(
    col("raw_name"),
    regexp_extract(trim(col("raw_name")), r"([A-Za-z]+)", 1).alias("first_word"),
).show(truncate=False)

# like — SQL LIKE pattern (% = wildcard)
df.filter(col("raw_name").like("%Smith%")).show(truncate=False)

# contains, startswith, endswith
df.filter(col("dept").contains("ing")).show()

## 4. Date Parsing & Formatting

> **Exam tip:** `to_date` / `to_timestamp` parse strings into date types. `date_format` formats a date into a string. The pattern letters follow Java's `SimpleDateFormat`.

In [ ]:
df2 = df.withColumn("hire_date", to_date(col("hire_date_str"), "yyyy-MM-dd")) \
        .withColumn("login_ts",  to_timestamp(col("login_ts_str"), "yyyy-MM-dd HH:mm:ss"))

df2.select("hire_date_str", "hire_date", "login_ts_str", "login_ts").printSchema()
df2.select("hire_date", "login_ts").show()

In [ ]:
# Extract parts of a date
df2.select(
    col("hire_date"),
    year("hire_date").alias("year"),
    month("hire_date").alias("month"),
    dayofmonth("hire_date").alias("day"),
    dayofweek("hire_date").alias("dow"),   # 1=Sun, 7=Sat
    quarter("hire_date").alias("quarter"),
    hour("login_ts").alias("login_hour"),
).show()

## 5. Date Arithmetic

In [ ]:
today = current_date()

df2.select(
    col("hire_date"),
    datediff(today, col("hire_date")).alias("days_employed"),
    months_between(today, col("hire_date")).cast("int").alias("months_employed"),
    date_add(col("hire_date"), 90).alias("probation_end"),
    date_sub(col("hire_date"), 7).alias("week_before_hire"),
    date_format(col("hire_date"), "dd/MM/yyyy").alias("eu_format"),
    date_format(col("hire_date"), "MMMM yyyy").alias("month_name"),
    trunc(col("hire_date"), "MM").alias("first_of_month"),
    date_trunc("hour", col("login_ts")).alias("hour_bucket"),
).show(truncate=False)

---

## String & Date Functions Quick Reference

| Category | Functions |
|---|---|
| Case | `upper`, `lower`, `initcap` |
| Whitespace | `trim`, `ltrim`, `rtrim` |
| Extraction | `substring`, `split`, `regexp_extract` |
| Replacement | `regexp_replace`, `translate`, `overlay` |
| Concat | `concat`, `concat_ws`, `format_string` |
| Search | `contains`, `startswith`, `endswith`, `like`, `instr`, `locate` |
| Length | `length`, `char_length` |
| Date parse | `to_date`, `to_timestamp`, `unix_timestamp`, `from_unixtime` |
| Date parts | `year`, `month`, `dayofmonth`, `hour`, `minute`, `second`, `quarter`, `dayofweek` |
| Date arith | `date_add`, `date_sub`, `datediff`, `months_between`, `add_months` |
| Date format | `date_format`, `trunc`, `date_trunc` |
| Current | `current_date`, `current_timestamp`, `now` |

## Day 10 Checklist

- [ ] Cleaned messy strings: `trim`, `upper`, `lower`, `regexp_replace`
- [ ] Split a string column into an array and extracted elements by index
- [ ] Used `to_date` and `to_timestamp` to parse strings
- [ ] Extracted `year`, `month`, `dayofmonth`, `hour` from a date
- [ ] Calculated `datediff` and `months_between`
- [ ] Formatted a date with `date_format` using a custom pattern

**Next:** Day 11 — Math & Collection Functions (arrays, maps, structs, explode)